# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."


get_food_calories_tool = function_tool(get_food_calories)


def get_food_water_content(food_item: str) -> str:
    """Get the approximate water content of common foods per 100 grams."""
    water_content_per_100g = {
        "apple": 86.0,
        "banana": 75.0,
        "broccoli": 89.0,
        "almonds": 4.4,
    }

    food_key = food_item.strip().lower()
    if food_key not in water_content_per_100g:
        return f"I don't have water content data for {food_item} in my database."

    water_grams = water_content_per_100g[food_key]
    return (
        f"{food_item.strip().title()}: approximately {water_grams:g} g of water "
        f"per 100 g ({water_grams:g}% water by weight). Values vary by variety and preparation."
    )


get_food_water_content_tool = function_tool(get_food_water_content)

Let's test this out:

`get_food_calories` remains directly callable, and `get_food_calories_tool` is the wrapper registered with the agent.

In [5]:
get_food_calories('rice')

"I don't have calorie data for rice in my database. Try common foods like apple, chicken breast, or rice."

In [6]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie and water content information.
    You give concise answers.
    Use get_food_calories for calorie lookups and get_food_water_content for water content lookups.
    """,
    tools=[get_food_calories_tool, get_food_water_content_tool],)

In [7]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories total (medium banana ~105 cal + medium apple ~80 cal).  
Sizes vary, so adjust if your fruits differ.


Enforce tools use:

In [10]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories_tool],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories in total (105 cal for a medium banana + 80 cal for a medium apple).


In [9]:
get_food_water_content("apple")

'Apple: approximately 86 g of water per 100 g (86% water by weight). Values vary by variety and preparation.'